# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

In [ ]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [ ]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [ ]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [1]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def get_cat_fact():
    requests.get(CAT_API_URL)

def sequence_get():
    for _ in range(20):
        get_cat_fact()

def multithread_get():
  with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    future_to_fact = [executor.submit(get_cat_fact) for _ in range(20)]

print("Sekwencyjnie: ")
start_time_sequential = time.time()
sequence_get()
end_time_sequential = time.time()

print(f"Czas wykonania sekwencyjnego: {end_time_sequential - start_time_sequential:.4f} sekund")

print("Wielowątkowo: ")
start_time_multithreading = time.time()
multithread_get()
end_time_multithreading = time.time()

print(f"Czas wykonania wielowątkowego: {end_time_multithreading - start_time_multithreading:.4f} sekund")

print("\n--- Porównanie czasów ---")
time_sequential = end_time_sequential - start_time_sequential
time_multithreaded = end_time_multithreading - start_time_multithreading

speedup = time_sequential / time_multithreaded
print(f"Wielowątkowość była {speedup:.2f} razy szybsza niż sekwencyjna.")


Sekwencyjnie: 
Czas wykonania sekwencyjnego: 6.7237 sekund
Wielowątkowo: 
Czas wykonania wielowątkowego: 1.0876 sekund

--- Porównanie czasów ---
Wielowątkowość była 6.18 razy szybsza niż sekwencyjna.


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [2]:
import queue
import threading
import time

kolejka = queue.Queue()
LIMIT_LICZB = 10

def producent():
    for i in range(1, LIMIT_LICZB + 1):
        print(f"[Producent] Generuję: {i}")
        kolejka.put(i)
        time.sleep(0.1)

    kolejka.put(None)

def konsument_parzyste():
    while True:
        liczba = kolejka.get()

        if liczba is None:
            kolejka.put(None)
            break

        if liczba % 2 == 0:
            print(f"  [Konsument 1 - Parzyste] Pobrałem: {liczba}")
        else:
            kolejka.put(liczba)
            time.sleep(0.05)

def konsument_nieparzyste():
    while True:
        liczba = kolejka.get()

        if liczba is None:
            kolejka.put(None)
            break

        if liczba % 2 != 0:
            print(f"  [Konsument 2 - Nieparzyste] Pobrałem: {liczba}")
        else:
            kolejka.put(liczba)
            time.sleep(0.05)

watek_prod = threading.Thread(target=producent)
watek_kons1 = threading.Thread(target=konsument_parzyste)
watek_kons2 = threading.Thread(target=konsument_nieparzyste)

watek_prod.start()
watek_kons1.start()
watek_kons2.start()

watek_prod.join()
watek_kons1.join()
watek_kons2.join()

print("Program zakończył pracę.")

[Producent] Generuję: 1
  [Konsument 2 - Nieparzyste] Pobrałem: 1
[Producent] Generuję: 2
  [Konsument 1 - Parzyste] Pobrałem: 2
[Producent] Generuję: 3
  [Konsument 2 - Nieparzyste] Pobrałem: 3
[Producent] Generuję: 4
  [Konsument 1 - Parzyste] Pobrałem: 4
[Producent] Generuję: 5
  [Konsument 2 - Nieparzyste] Pobrałem: 5
[Producent] Generuję: 6
  [Konsument 1 - Parzyste] Pobrałem: 6
[Producent] Generuję: 7
  [Konsument 2 - Nieparzyste] Pobrałem: 7
[Producent] Generuję: 8
  [Konsument 1 - Parzyste] Pobrałem: 8
[Producent] Generuję: 9
  [Konsument 2 - Nieparzyste] Pobrałem: 9
[Producent] Generuję: 10
  [Konsument 1 - Parzyste] Pobrałem: 10
Program zakończył pracę.


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [3]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    start_num = 1
    end_num = 100000
    numbers_to_process = range(start_num, end_num + 1)

    cores = multiprocessing.cpu_count()
    print(f"Dostępnych rdzeni CPU: {cores}")

    start_time = time.time()

    with multiprocessing.Pool(processes=cores) as pool:
        pool.map(calculate_power_sum, numbers_to_process)

    end_time = time.time()

    print(f"Zakończono obliczanie sum potęg dla liczb od {start_num} do {end_num}.")
    print(f"Czas wykonania (wieloprocesowo): {end_time - start_time:.4f}s")

Dostępnych rdzeni CPU: 8
Zakończono obliczanie sum potęg dla liczb od 1 do 100000.
Czas wykonania (wieloprocesowo): 2.9480s
